# 🤖 Resume Matching AI — Member 3

This Colab notebook compares a resume with a job description and gives:
- Semantic similarity
- Skill matching score
- Final match score
- Matching skills
- Missing skills


In [ ]:
# STEP 1 — Install library
!pip -q install sentence-transformers


In [ ]:
# STEP 2 — Imports
import re
from sentence_transformers import SentenceTransformer, util

print("Libraries imported successfully!")


In [ ]:
# STEP 3 — Skill extractor
SKILLS = [
    "python","java","c++","c","javascript","typescript","html","css",
    "react","angular","node.js","node","django","flask","fastapi",
    "sql","mysql","postgresql","mongodb","oracle",
    "machine learning","deep learning","artificial intelligence","ai",
    "data science","data analysis","nlp","computer vision",
    "pandas","numpy","scikit-learn","tensorflow","pytorch","keras",
    "matplotlib","seaborn","power bi","tableau","excel",
    "git","github","docker","kubernetes","aws","azure","gcp","linux",
    "rest api","api","spring","spring boot","communication",
    "leadership","problem solving","teamwork"
]
SKILLS = sorted(set(SKILLS), key=len, reverse=True)

def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[/|,;:()\[\]{}]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def extract_skills(text):
    text = normalize_text(text)
    found = []
    for skill in SKILLS:
        pattern = r"(?<!\w)" + re.escape(skill) + r"(?!\w)"
        if re.search(pattern, text):
            found.append(skill)
    return sorted(set(found))

print("Skill extractor ready!")


In [ ]:
# STEP 4 — Load AI model
model = SentenceTransformer("all-MiniLM-L6-v2")
print("AI model loaded successfully!")


In [ ]:
# STEP 5 — Main matching function
def match_resume_to_job(resume_text, job_description):
    if not resume_text.strip():
        raise ValueError("Resume text is empty.")
    if not job_description.strip():
        raise ValueError("Job description is empty.")

    resume_skills = set(extract_skills(resume_text))
    job_skills = set(extract_skills(job_description))

    matching_skills = sorted(resume_skills & job_skills)
    missing_skills = sorted(job_skills - resume_skills)

    skill_score = (
        len(matching_skills) / len(job_skills) * 100
        if job_skills else 0
    )

    embeddings = model.encode(
        [resume_text, job_description],
        convert_to_tensor=True,
        normalize_embeddings=True
    )
    semantic_score = float(util.cos_sim(embeddings[0], embeddings[1]).item() * 100)
    semantic_score = max(0, min(100, semantic_score))

    # 70% AI semantic similarity + 30% skill matching
    final_score = semantic_score * 0.70 + skill_score * 0.30

    return {
        "match_score": round(final_score, 2),
        "semantic_score": round(semantic_score, 2),
        "skill_score": round(skill_score, 2),
        "resume_skills": sorted(resume_skills),
        "job_skills": sorted(job_skills),
        "matching_skills": matching_skills,
        "missing_skills": missing_skills
    }

print("Matching function ready!")


In [ ]:
# STEP 6 — Demo
resume_text = '''
Python developer with experience in Machine Learning, Pandas,
NumPy, SQL and Data Analysis. Worked on AI projects and used Git.
'''

job_description = '''
We are hiring a Python Developer with experience in Machine Learning,
Pandas, SQL, Docker and AWS. Knowledge of Git and REST API is preferred.
'''

result = match_resume_to_job(resume_text, job_description)

print("===== RESUME MATCHING RESULT =====")
print(f"Final Match Score : {result['match_score']}%")
print(f"Semantic Score    : {result['semantic_score']}%")
print(f"Skill Score       : {result['skill_score']}%")

print("\nMatching Skills:")
for skill in result["matching_skills"]:
    print("  ✅", skill.title())

print("\nMissing Skills:")
for skill in result["missing_skills"]:
    print("  ❌", skill.title())


In [ ]:
# STEP 7 — Put your own resume and job description here
my_resume = '''
Paste the extracted resume text here.
'''

my_job = '''
Paste the job description here.
'''

# After replacing the text, run:
my_result = match_resume_to_job(my_resume, my_job)
print(my_result)
